# Chapter 1: Molecular Input & Data Structures

## What you'll learn

- How to load molecule data from file formats and build <code>Structure</code> objects within the QDK
- How to run an SCF and DFT calculation end-to-end
- The QDK's shared vocabulary of typed objects — <code>Structure</code>, <code>Wavefunction</code>, <code>Orbitals</code> — and how they flow between pipeline steps
- What each algorithm consumes and returns — and how to avoid common type confusion errors
- How the Chemistry QDK connects with Qiskit, PennyLane, OpenFermion, and RDKit

## The molecule for this course

We use **stretched N₂** throughout — nitrogen with a bond length of 1.27 Å (vs. equilibrium ~1.10 Å). The stretched geometry introduces strong multi-reference character, meaning no single electronic configuration adequately describes the ground state. This makes it an ideal system for exercising the full QDK workflow.

In [ ]:
# Setup: imports and molecule path
from pathlib import Path
import numpy as np

import qdk_chemistry.plugins.pyscf  # Required for scf_solver

from qdk_chemistry.data import Structure
from qdk_chemistry.algorithms import available, create, print_settings
from qdk_chemistry.constants import ANGSTROM_TO_BOHR
from qdk_chemistry.utils import Logger

Logger.set_global_level(Logger.LogLevel.off)

N2_XYZ = Path("../examples/data/stretched_n2.structure.xyz")

## Part 1: Loading a molecular structure

`Structure` is the entry point for all chemistry in the QDK. It wraps atomic coordinates and exposes methods for visualization and conversion.

The most common input route is an XYZ file via `Structure.from_xyz_file()`. You can also build a `Structure` directly from coordinates in memory using `Structure(coordinates=coords, symbols=symbols)` — useful when working programmatically or integrating with RDKit (see the interoperability section at the end of this chapter).

**Note**: the constructor takes coordinates in **Bohr**. XYZ files are in Angstrom and are converted automatically by `from_xyz_file()`. In the code snippet Load stretched N₂ from the XYZ file. Then build the same molecule directly from coordinates (in Bohr) and confirm they match.

In [ ]:
# Load N₂ structure from file and from coordinates
structure = Structure.from_xyz_file(N2_XYZ)
print("Loaded from file:")
print(structure.to_xyz())

# Method 2: build directly from coordinates (in Bohr)
# 1.27 Å × ANGSTROM_TO_BOHR (≈ 1.889 Bohr/Å) = 2.4008 Bohr
N2_BOND_BOHR = 1.27 * ANGSTROM_TO_BOHR
structure_manual = Structure(
    coordinates=np.array([[0.0, 0.0, 0.0], [0.0, 0.0, N2_BOND_BOHR]]),
    symbols=["N", "N"]
)
print("\nBuilt from coordinates (Bohr input):")
print(structure_manual.to_xyz())

## Part 2: Running SCF with a minimal basis

SCF (self-consistent field) is the classical starting point for any quantum chemistry workflow. It produces:
- A **<a href="https://chem.libretexts.org/Bookshelves/Physical_and_Theoretical_Chemistry_Textbook_Maps/Advanced_Theoretical_Chemistry_(Simons)/06:_Electronic_Structure/6.03:_The_Hartree-Fock_Approximation" target="_blank">Hartree-Fock energy</a>** (the best single-determinant approximation to the ground state)
- A <strong>Wavefunction</strong> object containing the molecular orbitals

The `scf_solver` is created via the registry. Its `run()` method takes the structure plus calculation parameters. We will run SCF on stretched N₂ with the `sto-3g` basis. 

In [ ]:
# Run SCF with STO-3G basis
scf_solver = create("scf_solver")
E_hf_sto3g, wfn_sto3g = scf_solver.run(
    structure,
    charge=0,
    spin_multiplicity=1,
    basis_or_guess="sto-3g"
)

print(f"HF energy (sto-3g): {E_hf_sto3g:.6f} Hartree")

## Part 3: Exploring the Wavefunction and Orbitals objects

The `Wavefunction` returned by SCF is the primary data object that flows through the rest of the pipeline. It contains the molecular orbitals, electron counts, and (later) determinant expansions.

Let's explore what the STO-3G wavefunction contains. Note the number of occupied and virtual orbitals.

In [ ]:
# Explore the Wavefunction and Orbitals objects
num_alpha, num_beta = wfn_sto3g.get_total_num_electrons()
print(f"Total electrons: alpha={num_alpha}, beta={num_beta}")

# Orbital summary
orbitals = wfn_sto3g.get_orbitals()
print("\nOrbital summary (sto-3g):")
print(orbitals.get_summary())

## Part 4: Comparing basis sets

Basis set choice determines how many orbitals are in the problem. A larger basis produces more molecular orbitals, which increases the size of the one- and two-body integral tables, raises the computational cost of SCF, and increases the number of orbitals available for downstream steps.

The `cc-pVDZ` basis is a standard double-zeta basis that is significantly larger than STO-3G and more chemically descriptive. It is the default choice for the rest of this course. A comprehensive catalog of basis sets, exponents, and contraction coefficients is available at <a href="https://www.basissetexchange.org/" target="_blank">Basis Set Exchange</a>. Now let's run SCF again with `cc-pvdz`. Compare the HF energies and orbital counts from both runs.

In [ ]:
# Run SCF with cc-pVDZ and compare energies
scf_solver_dz = create("scf_solver")
E_hf_dz, wfn_dz = scf_solver_dz.run(
    structure,
    charge=0,
    spin_multiplicity=1,
    basis_or_guess="cc-pvdz"
)

print(f"HF energy (sto-3g):  {E_hf_sto3g:.6f} Hartree")
print(f"HF energy (cc-pvdz): {E_hf_dz:.6f} Hartree")
print(f"Basis set effect:    {E_hf_dz - E_hf_sto3g:.6f} Hartree")

In [ ]:
# Inspect cc-pVDZ orbital summary
orbitals_dz = wfn_dz.get_orbitals()
print("Orbital summary (cc-pvdz):")
print(orbitals_dz.get_summary())

# YOUR CODE: How many more orbitals does cc-pvdz produce vs sto-3g?
# print(f"Orbital count increase: {???} orbitals")

# Free Response Question: ch1-orbitals-uhlfrc

How many more orbitals does cc-pvdz produce vs sto-3g?

Note: you could solve this question by manually counting orbitals - but don't! Make sure to run both code cells, and use QDK functions to get the answer. 

## Expected Answer

['18']


# Multiple Choice Question: ch1-basisset-rajq65

The cc-pVDZ basis produces 28 MOs for N₂ vs 10 for STO-3G. What is the direct downstream consequence of this for a quantum workflow?

## Choices

A. The HF energy is less accurate with cc-pVDZ
B. The active space selector has more orbitals to choose from, but qubit count scales with active space size
C. Localization is not possible with larger basis sets
D. The SCF solver will always fail to converge with cc-pVDZ


## Part 5: SCF settings and DFT

The SCF solver's `method` setting takes `"hf"` for Hartree-Fock or a DFT functional name for density functional theory — for example, `"b3lyp"`, `"pbe"`, or `"m06-2x"`. Settings are passed as keyword arguments to `create()`.

Let's run a DFT calculation on N₂ with `cc-pvdz`. Fill in the `???` below with a common hybrid functional, then compare the DFT and HF energies. Which is lower, and why?

In [ ]:
# Run DFT and compare with Hartree-Fock
scf_dft = create("scf_solver", "pyscf", method="???") 
E_dft, wfn_dft = scf_dft.run(structure, charge=0, spin_multiplicity=1, basis_or_guess="cc-pvdz")

print(f"DFT energy (cc-pvdz): {E_dft:.6f} Hartree")
print(f"HF  energy (cc-pvdz): {E_hf_dz:.6f} Hartree")
print(f"Difference:            {E_dft - E_hf_dz:.6f} Hartree")

# Free Response Question: ch1-dft-code-c2g0dy

Finish the code snippet below by replacing the "???" with the correct keyword. Enter the DFT energy (in Hartree) that you get. 

## Expected Answer

['-109.479851']


## Typed objects and data flow

Every step in the Chemistry QDK pipeline produces and consumes strongly-typed objects. Understanding this map prevents the most common errors — passing an `Orbitals` object where a `Wavefunction` is expected, or forgetting to call `.get_orbitals()` before passing to a Hamiltonian constructor. The diagram below walks you through the most important component of the overall Microsoft Chemistry QDK. 

<img src="https://storage.googleapis.com/qbraid-articles-staging/q-course-microsoft-chemistry-qdk/ch1-quantum_workflow_19.jpg?X-Goog-Algorithm=GOOG4-RSA-SHA256&X-Goog-Credential=cloud-run-api%40qbraid-staging.iam.gserviceaccount.com%2F20260505%2Fauto%2Fstorage%2Fgoog4_request&X-Goog-Date=20260505T205515Z&X-Goog-Expires=3600&X-Goog-SignedHeaders=host&X-Goog-Signature=REDACTED">



**Key rules:**
- <code>orbital_localizer</code> and <code>active_space_selector</code> both take and return <code>Wavefunction</code> — not <code>Orbitals</code>
- <code>hamiltonian_constructor</code> takes <code>Orbitals</code> — call <code>.get_orbitals()</code> first
- <code>state_prep</code> takes <code>Wavefunction</code> (the CASCI/ASCI result), not <code>Orbitals</code>
- Core energy (<code>hamiltonian.get_core_energy()</code>) must be added to QPE raw energy to get the total


## Interoperability

The Chemistry QDK is designed to work alongside other quantum software stacks. The typed objects produced at each pipeline stage can be exported to — or built from — external libraries.

| External stack | Integration point | How |
|---|---|---|
| **Qiskit** | `QubitHamiltonian`, `Circuit`, circuit executor | Via optional plugin: `qubit_mapper` can produce a Qiskit `SparsePauliOp`; `qiskit_aer_simulator` runs circuits |
| **Cirq** | `Circuit` | Via QASM3 export: `circuit.get_qasm()` → `cirq.from_qasm()` |
| **PennyLane** | `Circuit` | Via QASM3 export: `circuit.get_qasm()` → PennyLane `from_qasm()` |
| **OpenFermion** | `Hamiltonian` | Export one- and two-body integrals via `get_one_body_integrals()` / `get_two_body_integrals()` |
| **RDKit** | `Structure` input | Build `Structure` from SMILES via `Structure(coordinates=coords, symbols=symbols)` |

Qiskit is not a dependency of the Chemistry QDK, but rather an optional plugin. The section below presents a few examples that illustrate what interoperability looks like. These cells require `qdk-chemistry[qiskit-extras]` to be installed. They show what interoperability looks like and are not a turnkey workflow.

In [ ]:
# Export QDK QubitHamiltonian to Qiskit
import qdk_chemistry.plugins.qiskit
from qdk_chemistry.utils import compute_valence_space_parameters
from qiskit.quantum_info import SparsePauliOp

# Build a minimal active space from the STO-3G wavefunction (already computed in Part 2)
num_val_e, num_val_o = compute_valence_space_parameters(wfn_sto3g, charge=0)
wfn_valence = create("active_space_selector", "qdk_valence",
                     num_active_electrons=num_val_e,
                     num_active_orbitals=num_val_o).run(wfn_sto3g)
hamiltonian = create("hamiltonian_constructor").run(wfn_valence.get_orbitals())

# Map to qubits using the Qiskit Jordan-Wigner mapper
qubit_mapper = create("qubit_mapper", "qiskit", encoding="jordan-wigner")
qubit_hamiltonian = qubit_mapper.run(hamiltonian)

print(f"QDK QubitHamiltonian: {len(qubit_hamiltonian.pauli_strings)} Pauli strings")

# Construct a native Qiskit SparsePauliOp from the QDK representation
qiskit_op = SparsePauliOp(qubit_hamiltonian.pauli_strings, qubit_hamiltonian.coefficients)
print(f"Qiskit object type:   {type(qiskit_op).__name__}")
print(f"Num qubits:           {qiskit_op.num_qubits}")
print(f"\nFirst 3 Pauli terms:")
for pauli, coeff in list(zip(qiskit_op.paulis, qiskit_op.coeffs))[:3]:
    print(f"  {pauli}  coeff={coeff:.6f}")

Now, let's test that you fully understand this interoperability feature and its strengths. In addition to easily moving between QDK and other quantum software types, you could also bring in your fermionic or qubit Hamiltonian from other frameworks, and use the Microsoft QDK to calculate energies and properties. 

In the code snippet below, the Qiskit code defines the Qubit Hamiltonian for the hydrogen molecule, in the ground state, with a STO-3G basis. It also has been mapped with parity in mind, so it only needs 2 qubits. We see in this example how you can calculate the energy of the molecule using iterative QPE, which is unique to the Microsoft QDK. Right now, the QPE code is provided for you, but we will learn about each component in this course. 

Fill in the ```???```, and then answer the question below on qBook with the final energy you get. 

In [ ]:
# Import Qiskit SparsePauliOp into the QDK
import numpy as np                                                                                                                        
from qiskit.quantum_info import SparsePauliOp                                                                                             
from qiskit import QuantumCircuit, qasm3                                                                                                  
from qiskit.circuit.library import StatePreparation as QiskitStatePreparation                                                             
from qdk_chemistry.data import Circuit, QubitHamiltonian
from qdk_chemistry.algorithms import create

# H2 minimal basis 2-qubit Hamiltonian defined in Qiskit
h2_op = SparsePauliOp.from_list([
    ("II", -1.05342108),
    ("IZ",  0.39484436),
    ("XX",  0.18121046),
    ("ZI", -0.39484436),
    ("ZZ", -0.01124616),
])

# Bridge into QDK — fill in the blanks
qdk_h2 = QubitHamiltonian(
    pauli_strings="???",  
    coefficients="???",   
)

# Trial state: HF reference (|01⟩ in parity basis)
trial_vec = np.array([0.0, 1.0, 0.0, 0.0], dtype=complex)
qc = QuantumCircuit(2, name="trial")
qc.append(QiskitStatePreparation(trial_vec), [0, 1])
trial_circuit = Circuit(qasm3.dumps(qc))

# Run iterative QPE
iqpe = create("phase_estimation", "iterative",
              num_bits=6, evolution_time=np.pi/4, shots_per_bit=3)
simulator = create("circuit_executor", "qiskit_aer_simulator", seed=42)
evolution_builder = create("time_evolution_builder", "trotter")
circuit_mapper = create("controlled_evolution_circuit_mapper", "pauli_sequence")

result = iqpe.run(
    state_preparation=trial_circuit,
    qubit_hamiltonian=qdk_h2,
    circuit_executor=simulator,
    evolution_builder=evolution_builder,
    circuit_mapper=circuit_mapper,
)

energy = result.resolved_energy if result.resolved_energy is not None else result.raw_energy
NUCLEAR_REPULSION = 0.71510434 # hardcoded for now for H2 at 0.74 Å
print(f"Total ground state energy: {energy + NUCLEAR_REPULSION:.4f} Ha")

# Free Response Question: ch1-interop-wkqhy9

What is the final energy of the molecule calculated using iterative QPE?

## Expected Answer

['-1.159896']


## Summary


**Key patterns:**
```
# Load structure
structure = Structure.from_xyz_file(path)
structure = Structure(coordinates=coords_in_bohr, symbols=["N", "N"])

# SCF
energy, wfn = create("scf_solver").run(structure, charge=0, spin_multiplicity=1, basis_or_guess="cc-pvdz")

# Navigate types correctly
orbitals = wfn.get_orbitals()                             
# Wavefunction → Orbitals
hamiltonian = create("hamiltonian_constructor").run(orbitals)   
# Orbitals → Hamiltonian
total_energy = result.raw_energy + hamiltonian.get_core_energy()      
# always add core energy
```


In this chapter you:
- Mapped the full typed object pipeline: <code>Structure</code> → <code>Wavefunction</code> → <code>Orbitals</code> → <code>Hamiltonian</code> → <code>QubitHamiltonian</code> → <code>Circuit</code> → <code>QpeResult</code>
- Loaded <code>Structure</code> from XYZ file and from coordinates directly — noting that the constructor takes <strong>Bohr</strong>, while XYZ files use Angstrom
- Ran SCF with <code>sto-3g</code> and <code>cc-pvdz</code> and compared energies and orbital counts
- Ran DFT and compared against HF
- Saw how the QDK connects to RDKit, OpenFermion, Qiskit, and PennyLane

The `cc-pvdz` wavefunction (`wfn_dz`) is the starting point for Chapter 2.


You also tested your understanding of various tools and features by answering questions on qBook. 